In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from importlib import reload
from pathlib import Path

import xarrams as xrr
import ps
import carlee_tools as ct

from metpy.units import units
import metpy.constants as mpconstants
import metpy.calc as mpc

plt.style.use("cd_slides")

In [ ]:
# ps.backends.dask.connect_or_spawn_cluster()

In [ ]:
STORM = "sl_wk"
STORM_FULL_NAMES = {"ic_wk": "isolated_convection_wk", "sl_wk": "squall_line_wk"}

In [ ]:
rams_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/{STORM_FULL_NAMES[STORM]}/validation/fixed-True/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
rams_df = xrr.to_sounding_df(rams_ds.squeeze())
rams_df = xrr.calculate_sounding_derived_vars(rams_df)
rams_df

In [ ]:
reload(xrr.soundings)
reload(xrr)

# Also read one straight from the ramsin
ramsin_df = xrr.sounding_df_from_ramsin(
    f"/moonbow/cmdavis4/projects/bl_transport/rams_io/{STORM_FULL_NAMES[STORM]}/validation/fixed-True/RAMSIN"
)
ramsin_df = xrr.soundings.calculate_sounding_derived_vars(ramsin_df)
ramsin_df

In [ ]:
cm1_ds = xrr.read_cm1_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/validation_soundings/{STORM}/output"
            ).glob("cm1out_0*.nc")
        )
    )[:1]
)

reload(xrr.cm1)
reload(xrr)


def base_state_ds(cm1_ds):
    bs_variables = ["pi0", "th0", "prs0", "qv0"]
    # CM1 writes the four base-state variables at all times, so can pull them
    # from whenever
    t0_ds = cm1_ds.isel(time=0)
    # Rename them to match normal CM1 variable names
    t0_ds = t0_ds.drop([x[:-1] for x in bs_variables]).rename(
        {x: x[:-1] for x in bs_variables}
    )
    # Coerce to ramslike
    ds = xrr.cm1.coerce_to_ramslike(t0_ds)
    # Calculate derived thermodynamcis
    ds = xrr.calculate_thermodynamic_variables(ds)
    return ds


cm1_df = xrr.to_sounding_df(base_state_ds(cm1_ds))
cm1_df = xrr.calculate_sounding_derived_vars(cm1_df)
cm1_df

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("rams", rams_df), ("cm1", cm1_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

## VS ramsin

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("rams", rams_df), ("ramsin", ramsin_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

## Fixed vs not fixed

In [ ]:
fixed_dss = {}

for fixed in [False, True]:
    this_rams_ds = xrr.read_rams_output(
        sorted(
            list(
                Path(
                    f"/moonbow/cmdavis4/projects/bl_transport/rams_io/{STORM_FULL_NAMES[STORM]}/validation/fixed-{str(fixed)}/output"
                ).glob("a-A*.h5")
            )
        )[:1]
    )
    this_rams_df = xrr.to_sounding_df(this_rams_ds.squeeze())
    this_rams_df = xrr.calculate_sounding_derived_vars(this_rams_df)
    fixed_dss[fixed] = this_rams_df
ct.first_value(fixed_dss)

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("not fixed", fixed_dss[False]), ("fixed", fixed_dss[True])]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

# Squall line vs supercell

In [ ]:
sl_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/squall_line_wk/all_tracers/newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
sl_df = xrr.to_sounding_df(sl_ds.squeeze())
sl_df = xrr.calculate_sounding_derived_vars(sl_df)
sl_df

In [ ]:
sc_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/supercell_wk/all_tracers/newtracer_forcing-grant2014_dx-500_frqlite-2_stratwinds/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
sc_df = xrr.to_sounding_df(sc_ds.squeeze())
sc_df = xrr.calculate_sounding_derived_vars(sc_df)
sc_df

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("sc", sc_df), ("sl", sl_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

z_var = "PS"
# z_var = 'z'

for label, full_df in [("sc", sc_df), ("sl", sl_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df[z_var], label=label)
            # if z_var == 'PS':
            # ax.invert_yaxis()
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

# Supercell, with SL profile

## Vs squall line

In [ ]:
sc_sl_profile_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/from_cumulus/sc_wk/sl_profile/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
sc_sl_profile_df = xrr.to_sounding_df(sc_sl_profile_ds.squeeze())
sc_sl_profile_df = xrr.calculate_sounding_derived_vars(sc_sl_profile_df)
sc_sl_profile_df

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("sl", sl_df), ("sc_sl_profile", sc_sl_profile_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

## Vs original supercell

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("sc", sc_df), ("sc_sl_profile", sc_sl_profile_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

# IC, correct with BL vs not

In [ ]:
ic_paper_profile_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/isolated_convection_wk/all_tracers/newtracer_forcing-warmbubble4_deltax-500_frqlite-2_nobulku/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
ic_paper_profile_df = xrr.to_sounding_df(ic_paper_profile_ds.squeeze())
ic_paper_profile_df = xrr.calculate_sounding_derived_vars(ic_paper_profile_df)
ic_paper_profile_df

In [ ]:
ic_correct_w_bl_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/from_cumulus/ic_wk/correct_wk_wellmixed/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
ic_correct_w_bl_df = xrr.to_sounding_df(ic_correct_w_bl_ds.squeeze())
ic_correct_w_bl_df = xrr.calculate_sounding_derived_vars(ic_correct_w_bl_df)
ic_correct_w_bl_df

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [
    ("ic_paper", ic_paper_profile_df),
    ("ic_correct_w_bl", ic_correct_w_bl_df),
]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

# SL, correct with BL vs not

In [ ]:
sl_paper_profile_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/squall_line_wk/all_tracers/newtracer_blspinup-no_forcing-infinitecoldbubble4_deltax-500_frqlite-2/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
sl_paper_profile_df = xrr.to_sounding_df(sl_paper_profile_ds.squeeze())
sl_paper_profile_df = xrr.calculate_sounding_derived_vars(sl_paper_profile_df)
sl_paper_profile_df

In [ ]:
sl_correct_w_bl_ds = xrr.read_rams_output(
    sorted(
        list(
            Path(
                f"/moonbow/cmdavis4/projects/bl_transport/rams_io/from_cumulus/sl_wk/correct_wk_wellmixed_qv_fixed/output"
            ).glob("a-A*.h5")
        )
    )[:1]
)
sl_correct_w_bl_df = xrr.to_sounding_df(sl_correct_w_bl_ds.squeeze())
sl_correct_w_bl_df = xrr.calculate_sounding_derived_vars(sl_correct_w_bl_df)
sl_correct_w_bl_df

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [
    ("sl_paper", sl_paper_profile_df),
    ("sl_correct_w_bl", sl_correct_w_bl_df),
]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)

# Corrected SL vs original SC

In [ ]:
to_plot = [
    "TS",
    "dewpoint",
    "q_v",
    "RTS",
    "PS",
    "theta",
    "cape",
    "cin",
    "lcl",
    "lapse_rate_T_K_km",
    "lapse_rate_theta_K_km",
]
fig, axs = plt.subplots(
    nrows=len(to_plot), ncols=3, figsize=(12, 3 * len(to_plot)), layout="constrained"
)

for label, full_df in [("sl_correct_w_bl", sl_correct_w_bl_df), ("sc", sc_df)]:
    # Manually calculate lapse rate
    full_df["lapse_rate_T_K_km"] = (full_df["TS"].diff() / full_df["z"].diff()) * 1000
    full_df["lapse_rate_theta_K_km"] = (
        full_df["theta"].diff() / full_df["z"].diff()
    ) * 1000
    for var_ix, var in enumerate(to_plot):
        for ax_ix, z_cutoff in enumerate([99999, 5000, 500]):
            ax = axs[var_ix, ax_ix]
            df = full_df.loc[full_df["z"] <= z_cutoff]
            ax.plot(df[var], df["z"], label=label)
            ax.set_title(var)
            ct.plotting.clean_legend(ax, frameon=False)